# Use Case - Telemetry.ipynb

## Overview

This notebook provides an **operational telemetry dashboard** for services running on the
Sovereign Core platform. It is designed for **Managed Service Providers (MSPs), Billing Managers (IT Providers), and Platform Managers** who need
to understand how managed services are performing across their tenants — and to communicate
that picture clearly to stakeholders.

> **Note:** This is an example notebook intended to demonstrate what is possible with Sovereign Core metering data.
> The current demonstration runs on sample data. You can customise the charts, thresholds, and sections to suit your own needs,
> and evolve the notebook to include additional analyses — there are many more things you could do beyond what is shown here.

It uses **aggregated** and **grouped** metering data from the metrics-aggregator API.

If you need to model what tenants would be **charged** for their consumption,
use **`Use Case - Billing.ipynb`** instead.

---

### What this notebook produces

| # | Section | Chart | Question answered |
|---|---|---|---|
| 4.1 | KPI Summary | Indicator tiles + sparklines | What is the current value, trend, and range for each metric? |
| 4.2 | Resource Usage Trend | Synced multi-metric line chart | How has usage evolved — is it growing, stable, or declining? |
| 4.3 | Usage by Group — Total | Horizontal bar chart | Which tenants consumed the most across the full window? |
| 4.4 | Usage by Group — Trend | Multi-line time series | Is each tenant's usage growing or shrinking? |
| 4.5 | Usage Heatmap | Tenant × time heatmap | When is usage peaking, and which tenants drive it? |
| 4.6a | Telemetry Quality | Reporting coverage bar | How complete is the data — are there missing periods? |
| 4.6b | Anomaly Detection | Day-over-day bar chart | Which periods had unusually large jumps or drops? |

---

### Prerequisites

- This notebook runs out of the box using the **included sample data** — no deployment or API access needed.
- To use your own data, run **`Fetch - Usage Data.ipynb`** first to populate `data/aggregated/` and `data/grouped/`, or point the configuration (Section 2) at an existing data directory.

All charts are fully interactive — hover for exact values, click legend items to show/hide series,
drag to zoom, double-click to reset.

To adjust the anomaly sensitivity, change `DOD_THRESHOLD_PCT` in Section 2 (Configuration) and re-run.


## 1. Imports

In [1]:
import json
import math
import os
import pathlib

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dotenv import load_dotenv

## 2. Configuration

The fields below are pre-filled with the **demo environment** so the notebook runs out of the box
and always produces a consistent, repeatable set of charts.

| Default | Value |
|---|---|
| `APP_DOMAIN` | `apps.cluster.url.com` |
| `SERVICE_ID` | `servicebrokercore` |

**To use your own environment:** edit the values in the input fields and click **Apply Configuration**.
You can also set `RAW_DATA_PATH`, `AGGREGATED_DATA_PATH`, or `GROUPED_DATA_PATH` in `.env`
to override the data directories directly.

No secrets are loaded or printed here.

In [2]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Demo defaults ─────────────────────────────────────────────────────────────
DEMO_APP_DOMAIN = "apps.cluster.url.com"
DEMO_SERVICE_ID = "servicebrokercore"

# ── Load .env / .env.template for optional data path overrides ───────────────
if pathlib.Path(".env").exists():
    load_dotenv(".env", override=False)
else:
    load_dotenv(".env.template", override=False)

# ── Widgets ───────────────────────────────────────────────────────────────────
_style      = {"description_width": "130px"}
_layout     = widgets.Layout(width="480px")
_btn_layout = widgets.Layout(width="200px", height="36px", margin="12px 0 0 134px")

w_domain = widgets.Text(
    value=DEMO_APP_DOMAIN,
    description="APP_DOMAIN:",
    placeholder="e.g. apps.cluster.url.com",
    style=_style, layout=_layout,
)
w_service = widgets.Text(
    value=DEMO_SERVICE_ID,
    description="SERVICE_ID:",
    placeholder="e.g. cluster-as-a-service",
    style=_style, layout=_layout,
)
w_btn = widgets.Button(
    description="Apply Configuration",
    button_style="primary",
    icon="check",
    layout=_btn_layout,
)
w_out = widgets.Output()

# ── Global config state (populated on Apply) ──────────────────────────────────
app_domain = DEMO_APP_DOMAIN
service_id = DEMO_SERVICE_ID
RAW_DIR    = pathlib.Path(os.getenv("RAW_DATA_PATH")         or f"data/raw/{app_domain}/{service_id}")
AGG_DIR    = pathlib.Path(os.getenv("AGGREGATED_DATA_PATH")  or f"data/aggregated/{app_domain}/{service_id}")
GRP_DIR    = pathlib.Path(os.getenv("GROUPED_DATA_PATH")     or f"data/grouped/{app_domain}/{service_id}")
DOD_THRESHOLD_PCT = 30

def _apply(_):
    global app_domain, service_id, RAW_DIR, AGG_DIR, GRP_DIR
    app_domain = w_domain.value.strip() or DEMO_APP_DOMAIN
    service_id = w_service.value.strip() or DEMO_SERVICE_ID
    # .env path overrides take priority if set; otherwise derive from widget values
    RAW_DIR = pathlib.Path(os.getenv("RAW_DATA_PATH")        or f"data/raw/{app_domain}/{service_id}")
    AGG_DIR = pathlib.Path(os.getenv("AGGREGATED_DATA_PATH") or f"data/aggregated/{app_domain}/{service_id}")
    GRP_DIR = pathlib.Path(os.getenv("GROUPED_DATA_PATH")    or f"data/grouped/{app_domain}/{service_id}")
    tag_domain  = "  (demo default)" if app_domain == DEMO_APP_DOMAIN else "  (custom)"
    tag_service = "  (demo default)" if service_id == DEMO_SERVICE_ID else "  (custom)"
    with w_out:
        clear_output(wait=True)
        print(f"\u2713 APP_DOMAIN  : {app_domain}{tag_domain}")
        print(f"\u2713 SERVICE_ID  : {service_id}{tag_service}")
        print(f"\u2713 Anomaly threshold : \u00b1{DOD_THRESHOLD_PCT}% day-over-day\n")
        for label, d in [("raw", RAW_DIR), ("aggregated", AGG_DIR), ("grouped", GRP_DIR)]:
            files = list(d.glob("*.json"))
            if files:
                print(f"\u2713 {label:12s} \u2192 {d}  ({len(files)} files)")
            else:
                print(f"\u26a0 {label:12s} \u2192 {d}  (no JSON files found)")

w_btn.on_click(_apply)

display(
    widgets.VBox([
        widgets.HTML("<b style='font-size:13px'>Configure data source</b>"),
        w_domain,
        w_service,
        w_btn,
        w_out,
    ])
)

# Auto-apply defaults on first run so downstream cells work immediately
_apply(None)

## 3. Load Data

Loads all three data sources once and stores them in shared variables used by every chart section.

| Variable | Source directory | Contents |
|---|---|---|
| `raw_df` | `data/raw/` | One row per individual metering event |
| `agg_datasets` | `data/aggregated/` | Time-bucketed totals — one dataset per metric file |
| `grp_datasets` | `data/grouped/` | Time-bucketed totals split by tenant, workspace, or instance |

Run `Fetch - Usage Data.ipynb` first if any directory shows `⚠ no JSON files found`.

In [3]:
# ── Raw data ──────────────────────────────────────────────────────────────────
records = []
for path in sorted(RAW_DIR.glob("*.json")):
    with open(path) as f:
        records.extend(json.load(f).get("meteredUsage", []))

raw_df = pd.DataFrame(records)
if not raw_df.empty:
    raw_df.drop_duplicates(subset="id", keep="first", inplace=True)
    raw_df["startTimestamp"] = pd.to_datetime(raw_df["startTimestamp"], format="mixed", utc=True)
    raw_df["endTimestamp"]   = pd.to_datetime(raw_df["endTimestamp"],   format="mixed", utc=True)
    raw_df["usageQuantity"]  = pd.to_numeric(raw_df["usageQuantity"])
    raw_df["tenantLabel"]    = raw_df["tenantId"].replace("", "(unset)")
    raw_df["date"]           = raw_df["startTimestamp"].dt.floor("D")
    print(f"✓ raw_df        {len(raw_df):>6} records  metrics={sorted(raw_df['metricId'].unique())}")
else:
    print("⚠ raw_df — no records")


# ── Aggregated data ───────────────────────────────────────────────────────────
def _load_aggregated(directory: pathlib.Path) -> list:
    datasets = []
    for path in sorted(directory.glob("*.json")):
        with open(path) as f:
            data = json.load(f)
        periods = data.get("aggregatedMeteredUsagePeriods", [])
        if not periods:
            continue
        params = data.get("params", {})
        d = pd.DataFrame(periods)
        d["periodStart"]    = pd.to_datetime(d["periodStart"], unit="ms", utc=True)
        d["periodEnd"]      = pd.to_datetime(d["periodEnd"],   unit="ms", utc=True)
        d["periodQuantity"] = pd.to_numeric(d["periodQuantity"])
        datasets.append({
            "label":      path.stem,
            "metricId":   params.get("metricId", "unknown"),
            "transform":  params.get("transform", "unknown"),
            "groupByHrs": params.get("groupByHrs", 0),
            "df":         d,
        })
    return datasets


agg_datasets = _load_aggregated(AGG_DIR)
print(f"✓ agg_datasets  {len(agg_datasets):>6} files    metrics={[ds['metricId'] for ds in agg_datasets]}")


# ── Grouped data ──────────────────────────────────────────────────────────────
grp_datasets = []
for path in sorted(GRP_DIR.glob("*.json")):
    with open(path) as f:
        data = json.load(f)
    periods = data.get("aggregatedMeteredUsagePeriods", [])
    if not periods:
        continue
    params = data.get("params", {})
    d = pd.DataFrame(periods)
    d["periodStart"]    = pd.to_datetime(d["periodStart"], unit="ms", utc=True)
    d["periodEnd"]      = pd.to_datetime(d["periodEnd"],   unit="ms", utc=True)
    d["periodQuantity"] = pd.to_numeric(d["periodQuantity"])
    if "groupTenantId" in d.columns:
        d["groupTenantId"] = d["groupTenantId"].fillna("").replace("", "(anonymous)")
    if "groupId" in d.columns:
        d["groupLabel"] = d["groupId"].str.replace(r"^::", "(anonymous)::", regex=True)
    else:
        d["groupLabel"] = "(all)"
    total_groups = data.get("totalGroups", d["groupId"].nunique() if "groupId" in d.columns else 1)
    grp_datasets.append({
        "label":       path.stem,
        "metricId":    params.get("metricId", "unknown"),
        "transform":   params.get("transform", "unknown"),
        "groupByHrs":  params.get("groupByHrs", 0),
        "groupBy":     params.get("groupBy", "unknown"),
        "totalGroups": total_groups,
        "df":          d,
    })

print(f"✓ grp_datasets  {len(grp_datasets):>6} files    metrics={[ds['metricId'] for ds in grp_datasets]}")


# ── Shared helpers for grouped-data charts ────────────────────────────────────
_agg_fn = {
    'sum': ('sum', 'Total'),
    'avg': ('mean', 'Average'),
    'max': ('max', 'Peak'),
    'min': ('min', 'Floor'),
}

_y_labels = {
    'sum': 'Total quantity',
    'avg': 'Average quantity',
    'max': 'Peak quantity',
    'min': 'Floor quantity',
}


def _short_label(gid: str, group_by: str) -> str:
    """Shorten a composite group ID for chart axis labels.

    groupByTenant  ::tenantId:workspaceId:instanceId  →  last 8 chars of tenantId
    groupByWorkspace / other                          →  last 8 chars of gid
    """
    if not gid or gid == '(anonymous)':
        return '(anon)'
    if group_by == 'groupByTenant':
        # format is  ::tenantId:workspaceId:instanceId  — tenant is segment [2]
        parts = gid.split(':')
        tenant_part = next((p for p in parts if p), gid)
        return tenant_part[-8:] if len(tenant_part) > 8 else tenant_part
    return gid[-8:] if len(gid) > 8 else gid


✓ raw_df         36394 records  metrics=['api_calls', 'instances', 'users']
✓ agg_datasets       3 files    metrics=['instances', 'users', 'api_calls']
✓ grp_datasets       9 files    metrics=['instances', 'instances', 'instances', 'users', 'users', 'users', 'api_calls', 'api_calls', 'api_calls']


## 4. Charts

### 4.1 Telemetry KPI Summary

### Computation

For each aggregated metric dataset, computes the most recent period value (`q.iloc[-1]`),
the historical average (`q.mean()`), and the peak (`q.max()`).
Each metric is rendered as a two-row tile: a `go.Indicator` showing the big number
with a delta vs average, and a filled-area sparkline underneath showing the full trend.
Tiles are arranged in up to 3 columns using `make_subplots`.


In [4]:
fig_41 = None
if not agg_datasets:
    print("⚠ No aggregated datasets — run Fetch - Usage Data.ipynb first.")
else:
    n     = len(agg_datasets)
    ncols = min(3, n)
    nrows = math.ceil(n / ncols)
    COLOURS = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A", "#19D3F3", "#FF6692", "#B6E880"]
    def _hex_rgba(hex_colour, alpha=0.15):
        h = hex_colour.lstrip("#")
        r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
        return f"rgba({r},{g},{b},{alpha})"
    specs       = []
    row_heights = []
    for _ in range(nrows):
        specs.append([{"type": "domain"} for _ in range(ncols)])
        specs.append([{"type": "xy"}     for _ in range(ncols)])
        row_heights.extend([0.60, 0.40])
    fig_41 = make_subplots(
        rows=nrows * 2,
        cols=ncols,
        specs=specs,
        vertical_spacing=0.04,
        horizontal_spacing=0.08,
        row_heights=row_heights,
    )
    for idx, ds in enumerate(agg_datasets):
        col           = (idx % ncols) + 1
        tile_row      = (idx // ncols) * 2 + 1
        sparkline_row = tile_row + 1
        colour        = COLOURS[idx % len(COLOURS)]
        d       = ds["df"].sort_values("periodStart")
        q       = d["periodQuantity"]
        current = round(float(q.iloc[-1]), 2)
        avg_val = round(float(q.mean()), 2)
        peak    = round(float(q.max()), 2)
        fig_41.add_trace(
            go.Indicator(
                mode="number+delta",
                value=current,
                delta={
                    "reference":   avg_val,
                    "relative":    False,
                    "valueformat": ".2f",
                    "increasing":  {"color": "#00CC96"},
                    "decreasing":  {"color": "#EF553B"},
                },
                title={
                    "text": (
                        f"<b>{ds['metricId']}</b><br>"
                        f"<span style='font-size:0.75em;color:gray'>"
                        f"{ds['transform']} · {ds['groupByHrs']}h buckets</span><br>"
                        f"<span style='font-size:0.7em;color:gray'>"
                        f"avg {avg_val} &nbsp;|&nbsp; peak {peak}</span>"
                    ),
                    "align": "center",
                },
                number={"font": {"size": 40, "color": colour}, "valueformat": ".2f"},
            ),
            row=tile_row, col=col,
        )
        fig_41.add_trace(
            go.Scatter(
                x=d["periodStart"],
                y=q,
                mode="lines",
                line={"color": colour, "width": 2},
                fill="tozeroy",
                fillcolor=_hex_rgba(colour),
                hovertemplate="%{x|%b %d}<br>%{y:.2f}<extra></extra>",
                showlegend=False,
            ),
            row=sparkline_row, col=col,
        )
        fig_41.update_xaxes(showticklabels=False, showgrid=False, zeroline=False, row=sparkline_row, col=col)
        fig_41.update_yaxes(showticklabels=False, showgrid=False, zeroline=False, row=sparkline_row, col=col)
    fig_41.update_layout(
        height=300 * nrows,
        title_text="Telemetry KPI Summary — latest value · avg · peak · trend",
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin={"t": 60, "b": 20, "l": 20, "r": 20},
    )


### Chart Guide

**Purpose:** Answers the first question an operator asks when opening a service report —
what is the current state of each metric, and is it above or below normal?

**Chart Attributes**

| | |
|---|---|
| **Chart type** | KPI indicator tiles with sparklines |
| **Big number** | Most recent period value |
| **Delta ▲▼** | Difference from historical average (green = above, red = below) |
| **Sparkline** | Full trend as a filled area — shape shows growth, stability, or spikes |

**Insights:** A red delta across all metrics simultaneously may indicate a platform-wide
issue. A single metric with a large green delta while others are flat suggests an unusual
workload on that specific resource.


In [5]:
if fig_41 is not None:
    fig_41.show()


### 4.2 Resource Usage Trend

### Computation

Builds one subplot row per metric using `make_subplots` with `shared_xaxes=True`.
For each metric, three layers are drawn:
- **Raw series** — actual `periodQuantity` per bucket as a line with markers
- **Rolling average** — smoothed trend using a 7-period (or shorter) rolling window
- **Stability band** — shaded ±1 standard deviation region around the mean

All panels share the same X axis so cross-metric events can be correlated by date.


In [6]:
fig_42 = None
if not agg_datasets:
    print("⚠ No aggregated datasets — run Fetch - Usage Data.ipynb first.")
else:
    COLOURS = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A", "#19D3F3", "#FF6692", "#B6E880"]
    n   = len(agg_datasets)
    fig_42 = make_subplots(
        rows=n, cols=1,
        shared_xaxes=True,
        subplot_titles=[
            f"{ds['metricId']} ({ds['transform']} / {ds['groupByHrs']}h)"
            for ds in agg_datasets
        ],
        vertical_spacing=0.06,
    )
    for i, ds in enumerate(agg_datasets, start=1):
        d      = ds["df"].sort_values("periodStart")
        colour = COLOURS[(i - 1) % len(COLOURS)]
        fig_42.add_trace(
            go.Scatter(
                x=d["periodStart"],
                y=d["periodQuantity"],
                mode="lines+markers",
                name=f"{ds['metricId']} ({ds['transform']})",
                line={"color": colour, "width": 2},
                marker={"size": 5},
                hovertemplate="%{x|%b %d %H:%M}<br>%{y:.3f}<extra></extra>",
            ),
            row=i, col=1,
        )
        window = min(7, len(d))
        if window >= 2:
            roll = d["periodQuantity"].rolling(window=window, min_periods=1).mean()
            fig_42.add_trace(
                go.Scatter(
                    x=d["periodStart"],
                    y=roll,
                    mode="lines",
                    name=f"{ds['metricId']} {window}-period avg",
                    line={"color": colour, "width": 2, "dash": "dash"},
                    opacity=0.6,
                    hovertemplate="%{x|%b %d %H:%M}<br>rolling avg: %{y:.3f}<extra></extra>",
                    showlegend=True,
                ),
                row=i, col=1,
            )
        mean_val = d["periodQuantity"].mean()
        std_val  = d["periodQuantity"].std()
        if std_val > 0:
            h = colour.lstrip("#")
            r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
            x_vals = d["periodStart"].tolist()
            upper  = [mean_val + std_val] * len(x_vals)
            lower  = [mean_val - std_val] * len(x_vals)
            fig_42.add_trace(
                go.Scatter(
                    x=x_vals + x_vals[::-1],
                    y=upper + lower[::-1],
                    fill="toself",
                    fillcolor=f"rgba({r},{g},{b},0.10)",
                    line={"color": "rgba(0,0,0,0)"},
                    hoverinfo="skip",
                    showlegend=False,
                ),
                row=i, col=1,
            )
        fig_42.update_yaxes(title_text=f"{ds['transform']}", row=i, col=1)
    fig_42.update_layout(
        height=300 * n,
        title_text="Resource Usage Trend — rolling avg + stability band (±1 std dev)",
        hovermode="x unified",
        showlegend=True,
        legend={"orientation": "h", "y": -0.08, "x": 0.5, "xanchor": "center"},
        margin={"b": 80},
    )


### Chart Guide

**Purpose:** Shows how resource usage is trending over the reporting window with
noise reduction (rolling average) and a variability context (stability band).

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Faceted line chart (one subplot per metric, shared X axis) |
| **Solid line** | Raw period values |
| **Dashed line** | Rolling average (smoothed trend) |
| **Shaded band** | ±1 standard deviation — normal operating range |

**Insights:** Points outside the shaded band are statistical outliers.
Drag to zoom on any panel — all panels zoom together for cross-metric correlation.
A consistently widening band means usage is becoming more variable over time.


In [7]:
if fig_42 is not None:
    fig_42.show()


### 4.3 Usage by Group — Total Consumption

### Computation

For the first grouped dataset, aggregates `periodQuantity` per group using the method
that matches the dataset's `transform` (`sum` → grand total, `avg` → mean of period averages,
`max` → highest peak, `min` → lowest floor).
Group IDs are shortened to 8-character labels for readability; full IDs are in the hover.
Groups are sorted ascending so the largest bar appears at the top.


In [8]:
fig_43 = None
if not grp_datasets:
    print('⚠ No grouped datasets — run Fetch - Usage Data.ipynb first.')
else:
    # ── To show all datasets, replace the next line with: for ds in grp_datasets:
    ds = grp_datasets[0]
    d  = ds['df'].copy()
    if 'groupLabel' not in d.columns:
        print(f'  {ds["label"]} — no groupId column, skipping')
    else:
        agg_method, agg_desc = _agg_fn.get(ds['transform'], ('sum', f'Total ({ds["transform"]})'))
        d['displayLabel'] = d['groupId'].apply(
            lambda gid: _short_label(gid, ds['groupBy'])
        ) if 'groupId' in d.columns else d['groupLabel']
        group_totals = (
            d.groupby(['displayLabel', 'groupLabel'])['periodQuantity']
            .agg(agg_method).reset_index()
            .sort_values('periodQuantity', ascending=True)
        )
        group_totals.columns = ['displayLabel', 'fullLabel', 'total']
        fig_43 = px.bar(
            group_totals, x='total', y='displayLabel', orientation='h', text='total',
            custom_data=['fullLabel'],
            title=f"{ds['metricId']} — {agg_desc} per group  ·  grouped by: {ds['groupBy']}",
            labels={'total': agg_desc, 'displayLabel': 'Group'},
            color_discrete_sequence=['#636EFA'],
        )
        fig_43.update_traces(
            texttemplate='%{x:.2f}', textposition='outside', cliponaxis=False,
            hovertemplate='<b>%{y}</b><br>Full ID: %{customdata[0]}<br>'
                          f'{agg_desc}: %{{x:.2f}}<extra></extra>',
        )
        n_groups = len(group_totals)
        fig_43.update_layout(
            plot_bgcolor='white',
            xaxis=dict(showgrid=True, gridcolor='#eee'),
            yaxis=dict(tickfont=dict(family='monospace', size=12)),
            height=max(300, 60 + n_groups * 50),
            margin=dict(l=140, r=100, t=80, b=40),
            showlegend=False,
        )


### Chart Guide

**Purpose:** Ranks all groups by total consumption across the full window — the attribution
view for billing conversations and chargeback analysis.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Horizontal bar chart |
| **X axis** | Total usage quantity (aggregation matches transform) |
| **Y axis** | Group (shortened label) |
| **Hover** | Full group ID + exact value |

**Insights:** A single dominant bar means one group accounts for a disproportionate share.
Very short bars may indicate idle or newly provisioned tenants.
> Showing the first dataset. To show all, change `ds = grp_datasets[0]` to `for ds in grp_datasets:`.


In [9]:
if fig_43 is not None:
    fig_43.show()


### 4.4 Usage by Group — Trend Over Time

### Computation

For the first grouped dataset, sorts periods chronologically and plots one line per group.
Group IDs are shortened to readable labels; the full ID is preserved in hover via `customdata`.
The Y axis label is derived from the `transform` field so it always reflects the correct
aggregation semantics (sum / avg / max / min).


In [10]:
fig_44 = None
if not grp_datasets:
    print('⚠ No grouped datasets — run Fetch - Usage Data.ipynb first.')
else:
    # ── To show all datasets, replace the next line with: for ds in grp_datasets:
    ds = grp_datasets[0]
    d  = ds['df'].sort_values('periodStart').copy()
    y_label = _y_labels.get(ds['transform'], f'Quantity ({ds["transform"]})')
    d['displayLabel'] = d['groupId'].apply(
        lambda gid: _short_label(gid, ds['groupBy'])
    ) if 'groupId' in d.columns else d['groupLabel']
    fig_44 = px.line(
        d, x='periodStart', y='periodQuantity', color='displayLabel',
        markers=True, custom_data=['groupLabel'],
        title=f"{ds['metricId']} — {y_label} · {ds['groupByHrs']}h  ·  grouped by: {ds['groupBy']}",
        labels={'periodStart':'Period start (UTC)','periodQuantity':y_label,'displayLabel':'Group'},
    )
    fig_44.update_traces(
        hovertemplate='<b>Full ID: %{customdata[0]}</b><br>%{x|%b %d}<br>'
                      f'{y_label}: %{{y:.2f}}<extra></extra>'
    )
    fig_44.update_layout(
        hovermode='x unified',
        legend=dict(orientation='h', yanchor='top', y=-0.20, xanchor='left', x=0,
                    title='Group', font=dict(family='monospace', size=11)),
        margin=dict(l=60, r=40, t=80, b=120),
    )


### Chart Guide

**Purpose:** Complements the total bar chart (4.3) by showing the direction of travel —
is each group growing, declining, or holding steady over the window?

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Multi-series line chart |
| **X axis** | Period start timestamp (UTC) |
| **Y axis** | Usage quantity — label reflects the transform |
| **Colour** | One line per group |
| **Hover** | Full group ID + exact date and value |

**Insights:** A rising line means that tenant's consumption is increasing — potential upsell
or a capacity concern. A line dropping to zero mid-window may indicate churn or a disruption.
Click a group name in the legend to hide/show it when one tenant dominates the scale.
> Showing the first dataset. To show all, change `ds = grp_datasets[0]` to `for ds in grp_datasets:`.


In [11]:
if fig_44 is not None:
    fig_44.show()


### 4.5 Usage Heatmap

### Computation

Filters to tenant-grouped datasets (`groupBy=groupByTenant`).
For the first matching dataset, pivots `periodQuantity` into a `(tenant × time)` matrix
using `pivot_table`. Missing cells are filled with zero.
Full tenant IDs are carried via `customdata` (broadcast across each row) so hover always
shows the unabbreviated ID regardless of which cell is pointed at.


In [12]:
fig_45 = None
if not grp_datasets:
    print('⚠ No grouped datasets — run Fetch - Usage Data.ipynb first.')
else:
    tenant_datasets = [ds for ds in grp_datasets if ds['groupBy'] == 'groupByTenant']
    if not tenant_datasets:
        print('⚠ No groupByTenant datasets found — heatmap requires tenant-grouped data.')
    else:
        # ── To show all datasets, replace the next line with: for ds in tenant_datasets:
        ds = tenant_datasets[0]
        d = ds["df"].sort_values("periodStart").copy()
        # Short display label for Y axis
        d["displayLabel"] = d["groupId"].apply(
        lambda gid: _short_label(gid, "groupByTenant")
        ) if "groupId" in d.columns else d["groupLabel"]
        # Full tenant ID: one definitive value per displayLabel
        # Use groupTenantId if present, else fall back to groupId stripped of trailing colons
        if "groupTenantId" in d.columns:
            _full_id_src = d["groupTenantId"].fillna("").replace("", "(anonymous)")
        else:
            _full_id_src = d["groupId"].str.rstrip(":").fillna("(anonymous)")
        # Build displayLabel → fullTenantId mapping (one entry per group)
        _id_map = (
        d.assign(fullTenantId=_full_id_src)
        .groupby("displayLabel")["fullTenantId"]
        .first()
        .to_dict()
        )
        pivot_val = (
        d.pivot_table(
        index="displayLabel",
        columns="periodStart",
        values="periodQuantity",
        aggfunc="sum",
        )
        .fillna(0)
        )
        col_labels = [t.strftime("%b %d") for t in pivot_val.columns]
        y_labels   = pivot_val.index.tolist()
        # customdata: same shape as pivot_val — every cell in a row gets the same full tenant ID
        import numpy as np
        n_cols      = len(col_labels)
        customdata  = np.array(
        [[_id_map.get(lbl, lbl)] * n_cols for lbl in y_labels],
        dtype=object,
        )
        max_label_len = max(len(l) for l in y_labels) if y_labels else 10
        fig_45 = go.Figure(go.Heatmap(
        z=pivot_val.values,
        x=col_labels,
        y=y_labels,
        customdata=customdata,
        colorscale="Blues",
        text=[[f"{v:.2f}" for v in row] for row in pivot_val.values],
        texttemplate="%{text}",
        hovertemplate=(
        "<b>Tenant ID:</b> %{customdata}<br>"
        "<b>Period:</b> %{x}<br>"
        f"<b>Qty ({ds['transform']}):</b> %{{z:.2f}}"
        "<extra></extra>"
        ),
        colorbar=dict(title=f"Qty ({ds['transform']})"),
        ))
        fig_45.update_layout(
        title=(
        f"{ds['metricId']} — usage heatmap (tenant × time)\n"
        f"source: {ds['label']}"
        ),
        xaxis=dict(title="Period start (UTC)", tickangle=-45),
        yaxis=dict(
        title="Tenant",
        tickfont=dict(family="monospace", size=12),
        automargin=True,
        ),
        margin=dict(l=max(120, max_label_len * 10), r=80, t=80, b=80),
        plot_bgcolor="white",
        )


### Chart Guide

**Purpose:** Makes timing patterns immediately visible — which days are platform-wide peaks,
which tenants are active simultaneously, and which groups are consistently quiet.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Annotated heatmap |
| **X axis** | Period start date |
| **Y axis** | Tenant (short label) |
| **Colour** | Usage quantity — darker = higher value |
| **Hover** | Full tenant ID + exact date and quantity |

**Insights:** A dark column means all tenants had high usage on that day — a platform peak.
A dark cell in a single row is a tenant-specific spike. A consistently light row is a
low-usage or idle tenant worth investigating.
> A cell with value 0 may mean zero usage or no metering events received. See section 4.6.
> Showing the first tenant dataset. To show all, change `ds = tenant_datasets[0]` to `for ds in tenant_datasets:`.


In [13]:
if fig_45 is not None:
    fig_45.show()


### 4.6 Telemetry Quality + Anomalies

#### 4.6a — Reporting Coverage

### Computation

For each aggregated dataset, computes the expected number of time buckets given the query
window and `groupByHrs`, then counts actual received periods.
The gap (`expected − received`) is the number of missing buckets.
A summary table is printed first, then a stacked bar shows received (blue) vs missing (red)
per metric in a single figure.


In [14]:
fig_46a = None
# 6.1  Reporting coverage: expected vs received periods per dataset
if not agg_datasets:
    print("⚠ No aggregated datasets — run Fetch - Usage Data.ipynb first.")
else:
    coverage_rows = []
    for ds in agg_datasets:
        d = ds["df"].sort_values("periodStart")
        window_hrs = (
            (d["periodEnd"].max() - d["periodStart"].min()).total_seconds() / 3600
        )
        bucket   = ds["groupByHrs"] or window_hrs
        expected = max(1, math.ceil(window_hrs / bucket))
        received = len(d)
        gaps     = max(0, expected - received)
        coverage_pct = round(received / expected * 100, 1) if expected > 0 else 0
        coverage_rows.append({
            "Metric":        ds["metricId"],
            "Transform":     ds["transform"],
            "Bucket (h)":    ds["groupByHrs"],
            "Expected":      expected,
            "Received":      received,
            "Gaps":          gaps,
            "Coverage %":    coverage_pct,
        })
    coverage_df = pd.DataFrame(coverage_rows)
    print("Reporting coverage summary:")
    display(coverage_df)
    # Bar chart: received vs expected, gap shown in a separate colour
    fig_46a = go.Figure()
    fig_46a.add_trace(go.Bar(
        name="Received",
        x=coverage_df["Metric"],
        y=coverage_df["Received"],
        marker_color="#636EFA",
        text=coverage_df["Received"],
        textposition="inside",
    ))
    fig_46a.add_trace(go.Bar(
        name="Missing (gaps)",
        x=coverage_df["Metric"],
        y=coverage_df["Gaps"],
        marker_color="#EF553B",
        text=coverage_df["Gaps"],
        textposition="inside",
    ))
    fig_46a.update_layout(
        barmode="stack",
        title="Telemetry quality — reporting coverage (received vs missing periods)",
        xaxis_title="Metric",
        yaxis_title="Period count",
        legend_title="Status",
        plot_bgcolor="white",
        margin=dict(t=80, b=60),
    )


Reporting coverage summary:


,Metric,Transform,Bucket (h),Expected,Received,Gaps,Coverage %
0,instances,avg,24,14,13,1,92.9
1,users,avg,24,14,13,1,92.9
2,api_calls,sum,24,14,14,0,100.0


### Chart Guide

**Purpose:** Validates data completeness before using telemetry for billing or capacity
decisions. Missing periods mean gaps in the metering record.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Stacked bar chart |
| **X axis** | Metric |
| **Y axis** | Period count |
| **Blue segment** | Periods received (contain metering events) |
| **Red segment** | Periods missing (expected but absent) |

**Insights:** A large red segment means significant reporting gaps — investigate
whether the metering agent was down or the service genuinely had no activity.
100% blue across all metrics means the data is complete and trustworthy.


In [15]:
if fig_46a is not None:
    fig_46a.show()


#### 4.6b — Day-over-Day Anomaly Detection

### Computation

For the first aggregated dataset, computes `diff()` on `periodQuantity` to get the
period-over-period change, then calculates the percentage change relative to the prior value.
Bars exceeding `DOD_THRESHOLD_PCT` (set in Configuration) are coloured red (spike) or
orange (drop); normal bars are blue. A dotted average line provides context.
Only anomalous bars are labelled with their exact percentage change.


In [16]:
fig_46b = None

if not agg_datasets:
    print("⚠ No aggregated datasets — run Fetch - Usage Data.ipynb first.")

else:
    # ── To show all datasets, replace the next line with:
    # for ds in agg_datasets:
    ds = agg_datasets[0]

    d = ds["df"].sort_values("periodStart").copy()

    if len(d) < 2:
        print(
            f"  {ds['metricId']} — fewer than 2 periods, "
            "skipping anomaly view"
        )

    else:
        # ------------------------------------------------------------
        # Calculate day-over-day change
        # ------------------------------------------------------------
        d["prev"] = d["periodQuantity"].shift(1)

        d["dod_pct"] = (
            (
                (d["periodQuantity"] - d["prev"])
                / d["prev"].replace(0, np.nan)
            )
            * 100
        ).round(1)

        # ------------------------------------------------------------
        # Determine bar colour
        # ------------------------------------------------------------
        def _colour(row):
            if pd.isna(row["dod_pct"]):
                return "#aaa"

            if row["dod_pct"] > DOD_THRESHOLD_PCT:
                return "#EF553B"

            if row["dod_pct"] < -DOD_THRESHOLD_PCT:
                return "#FFA15A"

            return "#636EFA"

        d["barColour"] = d.apply(_colour, axis=1)

        # ------------------------------------------------------------
        # Labels
        # ------------------------------------------------------------
        d["dateLabel"] = d["periodStart"].dt.strftime("%b %d")

        n_anomalies = int(
            (d["dod_pct"].abs() > DOD_THRESHOLD_PCT).sum()
        )

        max_val = float(d["periodQuantity"].max())

        if not np.isfinite(max_val) or max_val <= 0:
            max_val = 1

        # Only label anomaly bars
        d["barLabel"] = d["dod_pct"].apply(
            lambda v: (
                f"{v:+.0f}%"
                if not pd.isna(v)
                and abs(v) > DOD_THRESHOLD_PCT
                else ""
            )
        )

        # ------------------------------------------------------------
        # Create figure
        # ------------------------------------------------------------
        fig_46b = go.Figure()

        fig_46b.add_trace(
            go.Bar(
                x=d["dateLabel"],
                y=d["periodQuantity"],
                marker_color=d["barColour"],
                customdata=d[["dod_pct"]],

                hovertemplate=(
                    "<b>%{x}</b><br>"
                    "Value: %{y:.2f}<br>"
                    "Day-over-day: %{customdata[0]:.1f}%"
                    "<extra></extra>"
                ),

                text=d["barLabel"],
                textposition="outside",
                cliponaxis=False,
                textfont=dict(
                    size=11,
                    color="#333"
                ),
            )
        )

        # ------------------------------------------------------------
        # Average line
        # ------------------------------------------------------------
        mean_val = float(d["periodQuantity"].mean())

        fig_46b.add_hline(
            y=mean_val,
            line_dash="dot",
            line_color="gray",
            annotation_text=f"avg {mean_val:.2f}",
            annotation_position="top right",
        )

        # ------------------------------------------------------------
        # Layout
        # ------------------------------------------------------------
        fig_46b.update_layout(
            title=(
                f"{ds['metricId']} ({ds['transform']}) — "
                f"day-over-day anomaly view "
                f"(threshold ±{DOD_THRESHOLD_PCT}% · "
                f"{n_anomalies} anomalies flagged)"
            ),

            xaxis=dict(
                title="Period",
                tickangle=-45,
                tickmode="linear",
            ),

            yaxis=dict(
                title=f"Quantity ({ds['transform']})",
                range=[0, max_val * 1.22],
            ),

            plot_bgcolor="white",
            showlegend=False,

            margin=dict(
                t=80,
                b=120,
                r=80,
                l=80,
            ),
        )

        # ------------------------------------------------------------
        # Colour key
        # ------------------------------------------------------------
        fig_46b.add_annotation(
            text=(
                "<span style='color:#EF553B'>"
                "▮ Spike"
                "</span> &gt;+{t}%&nbsp;&nbsp;&nbsp;"

                "<span style='color:#FFA15A'>"
                "▮ Drop"
                "</span> &gt;-{t}%&nbsp;&nbsp;&nbsp;"

                "<span style='color:#636EFA'>"
                "▮ Normal"
                "</span>"
            ).format(
                t=DOD_THRESHOLD_PCT
            ),

            xref="paper",
            yref="paper",

            x=0.5,
            y=-0.30,

            xanchor="center",
            yanchor="top",

            showarrow=False,

            font=dict(size=11),
            align="center",
        )

### Chart Guide

**Purpose:** Flags periods with unusually large jumps or drops that warrant investigation
before using the data in a customer-facing report.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Bar chart with diverging colours |
| **X axis** | Date |
| **Y axis** | Usage quantity |
| **Blue bars** | Normal — change within threshold |
| **Red bars** | Spike — increased by more than the threshold |
| **Orange bars** | Drop — decreased by more than the threshold |
| **Dotted line** | Period average — context for whether current level is high or low |

**Insights:** A single red bar following steady blue bars is an isolated spike worth
investigating. Multiple red/orange bars alternating indicate noisy, volatile data.
> Threshold is `DOD_THRESHOLD_PCT` (default ±30%), configurable in Section 2.
> Showing the first dataset. To show all, change `ds = agg_datasets[0]` to `for ds in agg_datasets:`.


In [17]:
if fig_46b is not None:
    fig_46b.show()
